# TinyChirp RMS distribution

Per-class clip-level RMS (in dBFS) across the whole dataset (train + val + test).
Mirrors the analysis done in the multi-chirp repo so we can compare level
distributions between datasets and decide on normalization / gain-aug strategy.

Bins of 2 dB from -70 to 0 dBFS.

In [ ]:
from pathlib import Path

import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

DATASET_DIR = Path("/home/nathan/Documents/tiny-chirp-microflow/dataset")
SPLITS = ("training", "validation", "testing")
CLASSES = ("target", "non_target")


def _rms_dbfs(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=np.float32)
    rms = np.sqrt(np.mean(x**2))
    return float(20.0 * np.log10(max(rms, 1e-10)))


def _scan(files: list[Path]) -> np.ndarray:
    out: list[float] = []
    for f in tqdm(files, desc=f"scan ({len(files)})", leave=False):
        try:
            x, _ = sf.read(str(f), dtype="float32", always_2d=False)
            if x.ndim > 1:
                x = x.mean(axis=-1)
            out.append(_rms_dbfs(x))
        except Exception:
            pass
    return np.array(out)


In [ ]:
# Scan every (split, class) and stash both per-split and aggregated arrays.
per_cell: dict[tuple[str, str], np.ndarray] = {}
for split in SPLITS:
    for cls in CLASSES:
        files = sorted((DATASET_DIR / split / cls).glob("*.wav"))
        per_cell[(split, cls)] = _scan(files)

per_class: dict[str, np.ndarray] = {
    cls: np.concatenate([per_cell[(s, cls)] for s in SPLITS]) for cls in CLASSES
}

# Per-split + per-class stats table.
hdr = f"{'split':<10s} {'class':<10s} {'n':>5s}  {'q05':>6s} {'q50':>6s} {'q95':>6s}  {'min':>6s} {'max':>6s}"
print(hdr)
print("-" * len(hdr))
for split in SPLITS:
    for cls in CLASSES:
        a = per_cell[(split, cls)]
        if len(a) == 0:
            continue
        print(
            f"{split:<10s} {cls:<10s} {len(a):>5d}  "
            f"{np.quantile(a, 0.05):>+6.1f} {np.median(a):>+6.1f} {np.quantile(a, 0.95):>+6.1f}  "
            f"{a.min():>+6.1f} {a.max():>+6.1f}"
        )

print()
print("Aggregated (all splits):")
print(hdr)
print("-" * len(hdr))
for cls in CLASSES:
    a = per_class[cls]
    print(
        f"{'all':<10s} {cls:<10s} {len(a):>5d}  "
        f"{np.quantile(a, 0.05):>+6.1f} {np.median(a):>+6.1f} {np.quantile(a, 0.95):>+6.1f}  "
        f"{a.min():>+6.1f} {a.max():>+6.1f}"
    )

In [ ]:
bins = np.arange(-70, 1, 2)

# Aggregated histogram: target vs non_target overlay.
fig, axes = plt.subplots(len(CLASSES), 1, figsize=(10, 3.0), sharex=True)
for ax, cls in zip(axes, CLASSES):
    a = per_class[cls]
    ax.hist(a, bins=bins, color="steelblue", edgecolor="black", alpha=0.85)
    ax.axvline(np.median(a), color="red", lw=1, ls="--", label=f"median {np.median(a):+.1f}")
    ax.set_ylabel(cls, rotation=0, ha="right", va="center")
    ax.legend(loc="upper left", fontsize=8, frameon=False)
axes[-1].set_xlabel("RMS (dBFS)")
fig.suptitle("TinyChirp: per-class RMS distribution (all splits)")
plt.tight_layout()
plt.show()

In [ ]:
# Per-split breakdown: rows = splits, cols = classes.
fig, axes = plt.subplots(len(SPLITS), len(CLASSES), figsize=(10, 5.0), sharex=True, sharey="row")
for i, split in enumerate(SPLITS):
    for j, cls in enumerate(CLASSES):
        ax = axes[i, j]
        a = per_cell[(split, cls)]
        ax.hist(a, bins=bins, color="steelblue", edgecolor="black", alpha=0.85)
        ax.axvline(np.median(a), color="red", lw=1, ls="--", label=f"median {np.median(a):+.1f}")
        if i == 0:
            ax.set_title(cls)
        if j == 0:
            ax.set_ylabel(split, rotation=0, ha="right", va="center")
        ax.legend(loc="upper left", fontsize=7, frameon=False)
for ax in axes[-1]:
    ax.set_xlabel("RMS (dBFS)")
fig.suptitle("TinyChirp: per-split, per-class RMS distribution")
plt.tight_layout()
plt.show()